# Beta vs SPY and SPY Equal Weight
Compare a single ticker's beta against SPY and the S&P 500 Equal Weight proxy (RSP) across two timeframes.


In [2]:
import yfinance as yf
import pandas as pd

# Configuration
ASSET_TICKER = "ASML"
MARKET_TICKER = "SPY"  # S&P 500 proxy
EQUAL_WEIGHT_TICKER = "RSP"  # S&P 500 Equal Weight proxy

TIMEFRAMES = [
    ("5y_monthly", "5y", "1mo"),
    ("2y_weekly", "2y", "1wk"),
]


def download_returns(tickers, period, interval):
    data = yf.download(tickers, period=period, interval=interval, progress=False)["Close"]
    returns = data.pct_change().dropna(how="all")
    return returns


def beta(asset_returns, market_returns):
    aligned = pd.concat([asset_returns, market_returns], axis=1).dropna()
    if aligned.empty:
        return float("nan")
    cov = aligned.iloc[:, 0].cov(aligned.iloc[:, 1])
    var = aligned.iloc[:, 1].var()
    return cov / var if var != 0 else float("nan")


def run_for_timeframe(label, period, interval):
    tickers = [ASSET_TICKER, MARKET_TICKER, EQUAL_WEIGHT_TICKER]
    returns = download_returns(tickers, period, interval)

    asset = returns[ASSET_TICKER]
    market = returns[MARKET_TICKER]
    equal_weight = returns[EQUAL_WEIGHT_TICKER]

    beta_vs_spy = beta(asset, market)
    beta_vs_rsp = beta(asset, equal_weight)

    print(f"\n=== {label} ===")
    print(f"{ASSET_TICKER} beta vs SPY: {beta_vs_spy:.4f}")
    print(f"{ASSET_TICKER} beta vs RSP: {beta_vs_rsp:.4f}")


for label, period, interval in TIMEFRAMES:
    run_for_timeframe(label, period, interval)



=== 5y_monthly ===
ASML beta vs SPY: 1.8299
ASML beta vs RSP: 1.5495

=== 2y_weekly ===
ASML beta vs SPY: 1.5383
ASML beta vs RSP: 1.3298
